# 01 — NumPy for Signals and Systems

NumPy is the layer everything else sits on. In a Signals and Systems course you will use it
for four things, over and over:

1. **Building time vectors** — the discrete grid your signal lives on.
2. **Generating signals** — sinusoids, exponentials, steps, impulses.
3. **Operating on whole signals at once** — shifting, scaling, windowing, convolving.
4. **Moving between time and frequency** — the FFT.

The single mental shift that matters: *a signal is an array, and an operation on a signal is
an operation on the whole array.* You almost never write a `for` loop over samples.

**How to use this notebook:** run every cell in order, then change numbers and re-run. The
exercises at the bottom have no solutions on purpose — that is the point.

---

## Contents

| § | Topic |
|---|---|
| 1 | Arrays, dtypes, and why vectorisation matters |
| 2 | Time vectors, sampling rate, and the Nyquist limit |
| 3 | Generating the standard signals |
| 4 | Indexing and slicing = time shifting and windowing |
| 5 | Broadcasting: many signals at once |
| 6 | Energy, power, RMS, and dB |
| 7 | Convolution — the LTI operation |
| 8 | Correlation, matched filtering, and time delay |
| 9 | The FFT: magnitude, phase, and frequency axes |
| 10 | Aliasing, seen directly |
| 11 | Linear algebra: the DFT matrix and state-space |
| 12 | Random numbers, noise, and reproducibility |
| 13 | Saving and loading |
| 14 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True, linewidth=100)
plt.rcParams["figure.figsize"] = (10, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("numpy", np.__version__)

---
## 1. Arrays, dtypes, and why vectorisation matters

A NumPy array is a fixed-size block of same-typed numbers. That constraint is exactly what
makes it fast: the loop over samples happens in compiled C, not in Python.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])

print("values   ", x)
print("shape    ", x.shape)      # (4,) -> 1-D, 4 samples
print("dtype    ", x.dtype)      # float64
print("itemsize ", x.itemsize, "bytes per sample")
print("nbytes   ", x.nbytes)

### dtype matters more than beginners expect

Integer arrays stay integers. If your signal is `int16` (as raw audio and most ADC captures
are), dividing or scaling can silently truncate or overflow.

In [ ]:
raw = np.array([100, 200, 300], dtype=np.int16)
print("int16 / 3      ->", raw / 3)          # numpy promotes to float64 here
print("int16 // 3     ->", raw // 3)         # floor division: truncated

# Overflow is silent, not an error:
big = np.array([32767], dtype=np.int16)
print("32767 + 1 as int16 ->", big + np.int16(1))

# So: convert ADC data to float as the very first step.
volts = raw.astype(np.float64) / 32768.0
print("as float volts ->", volts)

### Vectorisation is not a style preference

Compare a sample-by-sample loop against the array expression. Same maths, ~100x difference.

In [ ]:
n = 200_000
t = np.arange(n) / 1000.0

def loop_version(t):
    out = np.empty(len(t))
    for i in range(len(t)):
        out[i] = np.sin(2 * np.pi * 5 * t[i])
    return out

def vector_version(t):
    return np.sin(2 * np.pi * 5 * t)

# %timeit is an IPython magic -- it runs the line many times and reports the best result.
print("loop:")
%timeit -n 1 -r 3 loop_version(t)
print("vectorised:")
%timeit -n 10 -r 3 vector_version(t)

assert np.allclose(loop_version(t), vector_version(t))

---
## 2. Time vectors, sampling rate, and the Nyquist limit

Every discrete signal needs a time grid. Three quantities define it and each one determines
the others:

- $f_s$ — **sampling rate** in Hz (samples per second)
- $T_s = 1/f_s$ — **sampling period**, the gap between samples
- $N$ — number of samples, so duration $= N T_s$

The **Nyquist frequency** is $f_s/2$. Any content above it cannot be represented and will
fold back into your band as an alias. This is not a numerical artefact; it is a hard
information-theoretic limit.

In [ ]:
fs = 1000.0        # Hz
duration = 0.5     # seconds

# Two ways to build the grid. Know the difference.
t_arange = np.arange(0, duration, 1 / fs)          # excludes the endpoint
t_linspace = np.linspace(0, duration, int(fs * duration), endpoint=False)

print("arange   :", t_arange.size, "samples, last =", t_arange[-1])
print("linspace :", t_linspace.size, "samples, last =", t_linspace[-1])
print("identical:", np.allclose(t_arange, t_linspace))

print()
print("Nyquist frequency:", fs / 2, "Hz")
print("frequency resolution (1/duration):", 1 / duration, "Hz")

> **Pick `arange` or `linspace` deliberately.** `np.arange` with a float step can
> accumulate rounding error and occasionally emit one extra sample. `np.linspace(...,
> endpoint=False)` guarantees exactly `N` samples. For signal work, prefer `linspace` with
> `endpoint=False` — a periodic signal should *not* include both $t=0$ and $t=T$, because
> those are the same point on the cycle.

In [ ]:
def time_vector(fs, duration):
    """Return (t, N) for a duration-second capture at fs Hz, endpoint excluded."""
    n = int(round(fs * duration))
    return np.arange(n) / fs, n

t, N = time_vector(fs=8000, duration=0.01)
print(f"N = {N}, t[0] = {t[0]}, t[-1] = {t[-1]:.6f}, dt = {t[1] - t[0]:.6e}")

---
## 3. Generating the standard signals

These are the building blocks that appear in every problem set.

In [ ]:
fs = 500.0
t, N = time_vector(fs, 1.0)

# --- Continuous-time-style signals sampled on the grid --------------------
sine      = np.sin(2 * np.pi * 5 * t)
cosine    = np.cos(2 * np.pi * 5 * t)
phase_shift = np.sin(2 * np.pi * 5 * t + np.pi / 4)
decaying  = np.exp(-3 * t) * np.sin(2 * np.pi * 10 * t)
growing   = np.exp(0.8 * t)

# --- The complex exponential: the single most important signal in the course
# e^{j 2 pi f t} = cos(2 pi f t) + j sin(2 pi f t)
complex_exp = np.exp(1j * 2 * np.pi * 5 * t)

print("complex_exp dtype:", complex_exp.dtype)
print("real part == cosine:", np.allclose(complex_exp.real, cosine))
print("imag part == sine  :", np.allclose(complex_exp.imag, sine))
print("magnitude is 1     :", np.allclose(np.abs(complex_exp), 1.0))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 5), sharex=True)
ax[0, 0].plot(t, sine);        ax[0, 0].set_title("sin(2π·5t)")
ax[0, 1].plot(t, phase_shift, color="tab:orange")
ax[0, 1].plot(t, sine, alpha=0.35, color="tab:blue")
ax[0, 1].set_title("phase shift of π/4 (blue = original)")
ax[1, 0].plot(t, decaying, color="tab:green"); ax[1, 0].set_title("e^{-3t}·sin(2π·10t)")
ax[1, 1].plot(t, growing, color="tab:red");    ax[1, 1].set_title("e^{0.8t} (unstable)")
for a in ax[1]: a.set_xlabel("time [s]")
fig.tight_layout()

### Step, impulse, ramp, and rectangular pulse

In discrete time these are trivially easy, and writing them as reusable functions saves you
from re-deriving them in every homework.

In [ ]:
def unit_step(t, t0=0.0):
    """u(t - t0): 0 before t0, 1 from t0 onward."""
    return (t >= t0).astype(float)

def unit_impulse(n_samples, index=0):
    """Discrete delta delta[n - index]."""
    d = np.zeros(n_samples)
    d[index] = 1.0
    return d

def ramp(t, t0=0.0):
    """r(t - t0) = (t - t0)·u(t - t0)."""
    return (t - t0) * unit_step(t, t0)

def rect(t, start, stop):
    """Rectangular pulse, 1 on [start, stop)."""
    return ((t >= start) & (t < stop)).astype(float)

t, N = time_vector(fs=200, duration=2.0)

fig, ax = plt.subplots(1, 4, figsize=(13, 2.6))
ax[0].plot(t, unit_step(t, 0.5));       ax[0].set_title("u(t − 0.5)")
ax[1].plot(t, ramp(t, 0.5));            ax[1].set_title("r(t − 0.5)")
ax[2].plot(t, rect(t, 0.5, 1.2));       ax[2].set_title("rect on [0.5, 1.2)")
ax[3].stem(np.arange(15), unit_impulse(15, 4)); ax[3].set_title("δ[n − 4]")
ax[3].set_xlabel("n")
fig.tight_layout()

> **Note the comparison trick.** `t >= t0` produces a boolean array; `.astype(float)` turns
> `True/False` into `1.0/0.0`. This is the idiomatic way to build any piecewise signal, and
> it composes: `(t >= a) & (t < b)` for an interval, `|` for a union. Use `&` and `|`, not
> `and`/`or` — the Python keywords do not work element-wise.

### Discrete vs. continuous: use `stem`, not `plot`

When the signal is genuinely discrete-time (a sequence $x[n]$), plot it with `stem`.
Connecting samples with lines implies an interpolation you have not justified.

In [ ]:
n = np.arange(-5, 21)
x_n = (0.85 ** n) * (n >= 0)     # a^n · u[n], a classic discrete signal

fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
ax[0].stem(n, x_n); ax[0].set_title("x[n] = 0.85ⁿ·u[n]  (correct: stem)")
ax[1].plot(n, x_n, "-o", ms=3); ax[1].set_title("same data as a line (implies interpolation)")
for a in ax: a.set_xlabel("n")
fig.tight_layout()

---
## 4. Indexing and slicing = time shifting and windowing

Array slicing *is* the time-domain operation. Learn the mapping and half of the course's
manipulations become one-liners.

In [ ]:
x = np.arange(10.0)
print("x            ", x)
print("x[2:7]       ", x[2:7])       # window: samples 2..6
print("x[::-1]      ", x[::-1])      # time reversal: x[-n]
print("x[::2]       ", x[::2])       # downsample by 2 (no anti-alias filter!)
print("x[-3:]       ", x[-3:])       # last three samples

In [ ]:
def shift(x, k):
    """Delay by k samples (k > 0) or advance (k < 0), zero-padding the gap.

    Delay is x[n - k], so sample n of the output is sample n-k of the input.
    """
    out = np.zeros_like(x)
    if k > 0:
        out[k:] = x[:-k]
    elif k < 0:
        out[:k] = x[-k:]
    else:
        out[:] = x
    return out

sig = np.array([1., 2., 3., 4., 0., 0., 0., 0.])
print("original      ", sig)
print("delay by 2    ", shift(sig, 2))
print("advance by 2  ", shift(sig, -2))
print("reversed      ", sig[::-1])

> **Careful:** `np.roll` also shifts, but it *wraps around* rather than zero-padding. That is
> circular shift — correct for DFT-domain reasoning, wrong for a causal delay line.

In [ ]:
print("shift(sig, 2) :", shift(sig, 2))
print("np.roll(sig,2):", np.roll(sig, 2), "  <- note the wrap-around")

### Boolean masks: selecting by condition

Masks let you extract or modify samples that satisfy a condition — clipping, gating,
thresholding, removing dropouts.

In [ ]:
t, N = time_vector(fs=200, duration=1.0)
x = 1.6 * np.sin(2 * np.pi * 3 * t)

clipped = np.clip(x, -1.0, 1.0)                # hard limiter
gated = np.where(np.abs(x) > 0.8, x, 0.0)      # noise gate: keep only loud parts
loud_fraction = np.mean(np.abs(x) > 0.8)       # mean of a boolean = fraction True

print(f"fraction of samples above the gate threshold: {loud_fraction:.1%}")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(t, x, label="original", alpha=0.5)
ax.plot(t, clipped, label="clipped to ±1")
ax.plot(t, gated, label="gated at 0.8", lw=1)
ax.legend(loc="upper right"); ax.set_xlabel("time [s]")

---
## 5. Broadcasting: many signals at once

Broadcasting lets you compute a whole family of signals without a loop. This is how you
sweep a parameter — a frequency, a damping ratio, a filter order — in one expression.

The rule: NumPy compares shapes from the right; dimensions must be equal or one of them
must be 1, in which case it is stretched.

In [ ]:
t, N = time_vector(fs=500, duration=1.0)

freqs = np.array([2.0, 5.0, 9.0])          # shape (3,)
f_col = freqs[:, np.newaxis]               # shape (3, 1)  -- a column
t_row = t[np.newaxis, :]                   # shape (1, N)  -- a row

bank = np.sin(2 * np.pi * f_col * t_row)   # -> shape (3, N)

print("f_col shape:", f_col.shape)
print("t_row shape:", t_row.shape)
print("bank  shape:", bank.shape, " <- three signals, one expression")

fig, ax = plt.subplots(figsize=(10, 2.8))
for f, row in zip(freqs, bank):
    ax.plot(t, row, label=f"{f:g} Hz")
ax.legend(ncol=3); ax.set_xlabel("time [s]")

### A damped-response family

Second-order systems are the backbone of the course. Here is the step response of

$$H(s) = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

for several damping ratios $\zeta$, computed analytically and broadcast in one shot.

In [ ]:
t, N = time_vector(fs=500, duration=3.0)
wn = 6.0
zetas = np.array([0.1, 0.3, 0.7, 1.0])[:, np.newaxis]

# Underdamped closed form (valid for zeta < 1); zeta = 1 handled separately below.
wd = wn * np.sqrt(np.clip(1 - zetas**2, 0, None))
resp = 1 - np.exp(-zetas * wn * t) * (
    np.cos(wd * t) + np.where(wd > 0, zetas / np.where(wd > 0, np.sqrt(1 - zetas**2), 1), 0)
    * np.sin(wd * t)
)

fig, ax = plt.subplots(figsize=(10, 3))
for z, row in zip(zetas.ravel(), resp):
    ax.plot(t, row, label=f"ζ = {z}")
ax.axhline(1.0, color="k", ls=":", lw=1)
ax.set_xlabel("time [s]"); ax.set_ylabel("step response")
ax.legend(); ax.set_title(f"Second-order step response, ωₙ = {wn} rad/s")

---
## 6. Energy, power, RMS, and dB

Definitions you will be asked for constantly:

- **Energy**: $E = \sum_n |x[n]|^2$
- **Average power**: $P = \frac{1}{N}\sum_n |x[n]|^2$
- **RMS**: $\sqrt{P}$
- **dB** (power ratio): $10\log_{10}(P_1/P_2)$; for amplitudes, $20\log_{10}(A_1/A_2)$

The factor-of-two difference between the 10 and the 20 is the single most common mistake in
this material. Power gets 10, amplitude gets 20, because power goes as amplitude squared.

In [ ]:
def energy(x):
    return np.sum(np.abs(x) ** 2)

def power(x):
    return np.mean(np.abs(x) ** 2)

def rms(x):
    return np.sqrt(np.mean(np.abs(x) ** 2))

def db_power(p, ref=1.0):
    return 10 * np.log10(p / ref)

def db_amplitude(a, ref=1.0):
    return 20 * np.log10(np.abs(a) / ref)

t, N = time_vector(fs=1000, duration=1.0)
x = 2.0 * np.sin(2 * np.pi * 50 * t)

print(f"energy      {energy(x):.2f}")
print(f"power       {power(x):.4f}   (theory: A²/2 = {2.0**2 / 2})")
print(f"RMS         {rms(x):.4f}   (theory: A/√2 = {2.0 / np.sqrt(2):.4f})")
print(f"power in dB {db_power(power(x)):.2f} dB")

### Signal-to-noise ratio

SNR in dB is the workhorse metric for judging whether a filter helped.

In [ ]:
rng = np.random.default_rng(0)

clean = np.sin(2 * np.pi * 12 * t)
noise = rng.normal(0, 0.4, t.size)
noisy = clean + noise

def snr_db(clean, noisy):
    """SNR of `noisy` relative to the known `clean` reference."""
    err = noisy - clean
    return 10 * np.log10(np.sum(clean**2) / np.sum(err**2))

print(f"SNR = {snr_db(clean, noisy):.2f} dB")

# Sanity check: build a signal at an exact target SNR.
def add_noise_at_snr(clean, target_db, rng):
    sig_p = np.mean(clean ** 2)
    noise_p = sig_p / (10 ** (target_db / 10))
    return clean + rng.normal(0, np.sqrt(noise_p), clean.size)

for target in [20, 10, 0]:
    y = add_noise_at_snr(clean, target, rng)
    print(f"requested {target:>3} dB  ->  measured {snr_db(clean, y):6.2f} dB")

---
## 7. Convolution — the LTI operation

An LTI system is completely described by its impulse response $h$. The output for *any*
input is the convolution

$$y[n] = (x * h)[n] = \sum_k x[k]\, h[n-k]$$

`np.convolve` has three modes and choosing the wrong one is a common source of confusion:

| mode | output length | meaning |
|------|---------------|---------|
| `'full'` | `len(x) + len(h) - 1` | the complete mathematical result (default) |
| `'same'` | `len(x)` | centre slice, same length as input |
| `'valid'`| `len(x) - len(h) + 1` | only where the kernel fully overlaps |

In [ ]:
x = np.array([1., 2., 3., 4., 5.])
h = np.array([1., 1., 1.]) / 3        # 3-point moving average

print("full :", np.convolve(x, h, mode="full"))
print("same :", np.convolve(x, h, mode="same"))
print("valid:", np.convolve(x, h, mode="valid"))
print()
print("lengths:", len(x), "+", len(h), "- 1 =", len(x) + len(h) - 1)

In [ ]:
# A moving-average filter smoothing a noisy signal.
t, N = time_vector(fs=500, duration=1.0)
rng = np.random.default_rng(1)
clean = np.sin(2 * np.pi * 4 * t)
noisy = clean + rng.normal(0, 0.35, N)

for M in [5, 21, 61]:
    h = np.ones(M) / M
    smooth = np.convolve(noisy, h, mode="same")
    plt.plot(t, smooth, lw=1.2, label=f"M = {M}")

plt.plot(t, noisy, color="0.8", lw=0.7, zorder=0, label="noisy")
plt.plot(t, clean, "k--", lw=1, label="clean")
plt.legend(ncol=5, fontsize=8); plt.xlabel("time [s]")
plt.title("Moving average: longer kernel = smoother, but more delay and attenuation")

> **What the plot shows.** Longer kernels suppress more noise but also attenuate the signal
> and introduce delay. That trade-off — noise rejection vs. bandwidth vs. delay — is the
> central tension in filter design, and you are seeing it in three lines of code.

### Verifying the LTI properties numerically

Rather than trusting the algebra, check it.

In [ ]:
rng = np.random.default_rng(2)
x1 = rng.normal(size=40)
x2 = rng.normal(size=40)
h = rng.normal(size=7)
a, b = 2.5, -1.3

# Linearity: h*(a·x1 + b·x2) == a·(h*x1) + b·(h*x2)
lhs = np.convolve(a * x1 + b * x2, h)
rhs = a * np.convolve(x1, h) + b * np.convolve(x2, h)
print("linearity holds :", np.allclose(lhs, rhs))

# Commutativity: x*h == h*x
print("commutative     :", np.allclose(np.convolve(x1, h), np.convolve(h, x1)))

# Associativity: (x*h1)*h2 == x*(h1*h2)
h2 = rng.normal(size=5)
print("associative     :", np.allclose(
    np.convolve(np.convolve(x1, h), h2),
    np.convolve(x1, np.convolve(h, h2)),
))

# Time-invariance: delaying the input delays the output by the same amount.
d = 6
y = np.convolve(x1, h)
y_delayed_input = np.convolve(np.concatenate([np.zeros(d), x1]), h)
print("time-invariant  :", np.allclose(y_delayed_input[d:], y))

### Convolution *is* polynomial multiplication

A fact worth internalising: convolving coefficient sequences multiplies the corresponding
polynomials. This is why the transfer function of cascaded systems is the product of the
individual transfer functions.

In [ ]:
# (1 + 2z + 3z²)·(1 + z) = 1 + 3z + 5z² + 3z³
p1 = np.array([1., 2., 3.])
p2 = np.array([1., 1.])
print("convolve      :", np.convolve(p1, p2))
print("polymul       :", np.polynomial.polynomial.polymul(p1, p2))

---
## 8. Correlation, matched filtering, and time delay

Cross-correlation measures similarity as a function of lag. It looks like convolution but
without the flip — and it is how you find *when* something happened.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 1.0)
rng = np.random.default_rng(3)

# A short pulse hidden in noise at a known delay.
template = np.exp(-((t[:80] - 0.04) ** 2) / (2 * 0.008 ** 2)) * np.sin(2 * np.pi * 60 * t[:80])
true_delay_samples = 430

received = rng.normal(0, 0.8, N)
received[true_delay_samples:true_delay_samples + template.size] += template

# Cross-correlate. 'full' gives lags from -(N-1) to +(M-1).
corr = np.correlate(received, template, mode="full")
lags = np.arange(-len(template) + 1, len(received))
peak_lag = lags[np.argmax(corr)]

print(f"true delay      : {true_delay_samples} samples ({true_delay_samples / fs * 1000:.1f} ms)")
print(f"detected delay  : {peak_lag} samples ({peak_lag / fs * 1000:.1f} ms)")

fig, ax = plt.subplots(2, 1, figsize=(10, 4.5))
ax[0].plot(t, received, lw=0.6); ax[0].set_title("received signal (pulse invisible by eye)")
ax[0].set_xlabel("time [s]")
ax[1].plot(lags / fs, corr, lw=0.8)
ax[1].axvline(peak_lag / fs, color="r", ls="--", label=f"peak at {peak_lag / fs:.3f} s")
ax[1].set_title("cross-correlation with the template"); ax[1].set_xlabel("lag [s]")
ax[1].legend()
fig.tight_layout()

> **This is radar, sonar, GPS, and ultrasound in one cell.** You transmit a known waveform,
> correlate the echo against it, and the peak location gives you the round-trip time. The
> correlation also buys you processing gain: the pulse was invisible in the top panel and
> unmistakable in the bottom one.

In [ ]:
# Autocorrelation reveals periodicity -- useful for pitch and heart-rate detection.
t, N = time_vector(fs=500, duration=2.0)
periodic = np.sin(2 * np.pi * 7 * t) + 0.5 * np.sin(2 * np.pi * 14 * t)
noisy_periodic = periodic + np.random.default_rng(4).normal(0, 1.0, N)

ac = np.correlate(noisy_periodic, noisy_periodic, mode="full")
ac = ac[ac.size // 2:]              # keep non-negative lags
ac = ac / ac[0]                     # normalise so lag 0 == 1
lag_s = np.arange(ac.size) / 500.0

plt.plot(lag_s[:300], ac[:300])
plt.axvline(1 / 7, color="r", ls="--", label="expected period 1/7 s")
plt.xlabel("lag [s]"); plt.ylabel("normalised autocorrelation"); plt.legend()
plt.title("Autocorrelation finds the period even under heavy noise")

---
## 9. The FFT: magnitude, phase, and frequency axes

The DFT converts $N$ time samples into $N$ complex frequency bins. `np.fft` computes it
efficiently. The part students get wrong is the **frequency axis**, so build it with
`fftfreq` and never by hand.

For real-valued signals use `rfft`: the spectrum is conjugate-symmetric, so the negative
half carries no new information and `rfft` returns only bins $0 \ldots N/2$.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 1.0)          # 1 s -> 1 Hz resolution

x = (1.0 * np.sin(2 * np.pi * 50 * t)
     + 0.5 * np.sin(2 * np.pi * 120 * t + np.pi / 3)
     + 0.2 * np.sin(2 * np.pi * 300 * t))

X = np.fft.rfft(x)                   # complex spectrum, length N//2 + 1
freqs = np.fft.rfftfreq(N, d=1 / fs) # matching frequency axis in Hz

# Scale so a sinusoid of amplitude A reads as A.
mag = 2 * np.abs(X) / N
mag[0] /= 2                          # DC bin is not doubled
if N % 2 == 0:
    mag[-1] /= 2                     # nor is Nyquist

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t[:200], x[:200]); ax[0].set_title("time domain (first 200 ms)")
ax[0].set_xlabel("time [s]")
ax[1].plot(freqs, mag); ax[1].set_xlim(0, 400)
ax[1].set_title("magnitude spectrum"); ax[1].set_xlabel("frequency [Hz]")
for f, a in [(50, 1.0), (120, 0.5), (300, 0.2)]:
    ax[1].annotate(f"{a}", xy=(f, a), xytext=(f + 8, a + 0.05), fontsize=9)
fig.tight_layout()

### The amplitude-scaling rule, stated plainly

`np.fft.rfft` returns an *unnormalised* sum. To read amplitudes off the plot directly:

- divide by `N`
- multiply by `2` (because a real sinusoid splits its energy between $+f$ and $-f$)
- do **not** double the DC bin or the Nyquist bin — they have no mirror partner

If you only care about relative levels or you plot in dB, the scaling cancels and you can
skip it. If a homework question asks "what is the amplitude of the 50 Hz component", you
cannot.

In [ ]:
# Phase. Only meaningful where the magnitude is non-negligible -- elsewhere it is noise.
phase = np.angle(X)
significant = mag > 0.05 * mag.max()

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(freqs[significant], np.degrees(phase[significant]), "o", ms=5)
ax.set_xlim(0, 400); ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("phase [deg]")
ax.set_title("Phase at the significant bins only")
print("phase at 120 Hz:", np.degrees(phase[120]).round(1), "deg   (input had +60°, sine vs cosine reference differs by 90°)")

### Spectral leakage: why your peak is smeared

The DFT assumes your capture repeats forever. If the signal does not contain a whole number
of cycles in the window, the wrap-around creates a discontinuity, and the spectrum smears.
This is **leakage**, and a window function is the fix.

In [ ]:
fs, N = 1000.0, 1000
t = np.arange(N) / fs

exact = np.sin(2 * np.pi * 100 * t)      # exactly 100 cycles in the window
inexact = np.sin(2 * np.pi * 100.5 * t)  # 100.5 cycles -> discontinuity at the wrap

f = np.fft.rfftfreq(N, 1 / fs)

def spectrum_db(x, win=None):
    if win is not None:
        x = x * win
    X = np.abs(np.fft.rfft(x))
    return 20 * np.log10(X / X.max() + 1e-12)

window = np.hanning(N)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(f, spectrum_db(exact), label="100.0 Hz, no window (bin-centred)")
ax.plot(f, spectrum_db(inexact), label="100.5 Hz, no window (leaks)")
ax.plot(f, spectrum_db(inexact, window), label="100.5 Hz, Hann window")
ax.set_xlim(60, 140); ax.set_ylim(-120, 5)
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("dB")
ax.legend(fontsize=8); ax.set_title("Spectral leakage and the effect of windowing")

> **Read the plot carefully.** The blue trace falls off a cliff — that is what a perfectly
> bin-centred sinusoid looks like. Orange leaks energy across the entire band. Green (Hann
> window) trades a slightly wider main lobe for dramatically lower sidelobes. In practice you
> almost always window before taking an FFT of real data.

### The convolution theorem

Convolution in time is multiplication in frequency. This is both a proof technique and a
fast algorithm.

In [ ]:
rng = np.random.default_rng(5)
x = rng.normal(size=256)
h = rng.normal(size=64)

direct = np.convolve(x, h)                      # length 256 + 64 - 1 = 319

L = len(x) + len(h) - 1
nfft = 1 << (L - 1).bit_length()                # next power of two, for speed
via_fft = np.fft.irfft(np.fft.rfft(x, nfft) * np.fft.rfft(h, nfft), nfft)[:L]

print("max absolute difference:", np.max(np.abs(direct - via_fft)))
print("agree:", np.allclose(direct, via_fft))

> **Zero-padding to `nfft` is not optional.** Multiplying the DFTs computes *circular*
> convolution of length `nfft`. Padding both signals to at least `len(x) + len(h) - 1` makes
> the circular result equal the linear one. Forget this and you get time-aliased garbage
> wrapped around the start of your output.

---
## 10. Aliasing, seen directly

Sample a sinusoid above Nyquist and it comes back as a *different, lower* frequency. The
alias appears at $|f - k f_s|$ for the integer $k$ that lands the result in $[0, f_s/2]$.

In [ ]:
fs = 100.0                       # Nyquist = 50 Hz
t_s = np.arange(0, 0.4, 1 / fs)
t_fine = np.linspace(0, 0.4, 4000)   # a stand-in for "continuous"

fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for a, f in zip(ax, [20.0, 80.0, 105.0]):
    alias = abs(f - fs * round(f / fs))
    a.plot(t_fine, np.sin(2 * np.pi * f * t_fine), color="0.75", lw=1,
           label=f"true {f:g} Hz")
    a.plot(t_fine, np.sin(2 * np.pi * alias * t_fine), "g--", lw=1.2,
           label=f"alias {alias:g} Hz")
    a.plot(t_s, np.sin(2 * np.pi * f * t_s), "ro", ms=5, label=f"samples @ {fs:g} Hz")
    a.legend(loc="upper right", fontsize=8, ncol=3)
    a.set_ylabel(f"{f:g} Hz")
ax[-1].set_xlabel("time [s]")
fig.suptitle("The samples cannot tell the two frequencies apart", y=1.0)
fig.tight_layout()

> **The 20 Hz case is fine** (below Nyquist). **80 Hz aliases to 20 Hz** and **105 Hz aliases
> to 5 Hz** — and in each case the red dots sit exactly on *both* curves. No algorithm can
> undo this after the fact. The only defence is an analogue anti-aliasing filter before the
> ADC.

---
## 11. Linear algebra: the DFT matrix and state-space

`np.linalg` gives you the tools for state-space analysis, and writing the DFT as a matrix
makes it concrete that it is just a change of basis.

In [ ]:
def dft_matrix(N):
    """The N x N DFT matrix W, where X = W @ x."""
    n = np.arange(N)
    return np.exp(-2j * np.pi * np.outer(n, n) / N)

N = 16
W = dft_matrix(N)
x = np.random.default_rng(6).normal(size=N)

print("matrix DFT == np.fft.fft:", np.allclose(W @ x, np.fft.fft(x)))
print("W is (scaled) unitary   :", np.allclose(W.conj().T @ W, N * np.eye(N)))
print()
print("So the inverse is just the conjugate transpose over N:")
print("  reconstruction exact  :", np.allclose(W.conj().T @ (W @ x) / N, x))

In [ ]:
# State-space stability: eigenvalues of A decide everything.
# Continuous time: stable iff all Re(lambda) < 0.
A_c = np.array([[0.0, 1.0],
                [-4.0, -0.5]])
eig_c = np.linalg.eigvals(A_c)
print("continuous-time eigenvalues:", eig_c)
print("stable:", np.all(eig_c.real < 0))
print()

# Discrete time: stable iff all |lambda| < 1.
A_d = np.array([[0.9, 0.2],
                [-0.1, 0.8]])
eig_d = np.linalg.eigvals(A_d)
print("discrete-time eigenvalues:", eig_d, " magnitudes:", np.abs(eig_d).round(4))
print("stable:", np.all(np.abs(eig_d) < 1))

In [ ]:
# Controllability of (A, B): full-rank controllability matrix [B, AB, A²B, ...].
B = np.array([[0.0], [1.0]])
ctrb = np.hstack([np.linalg.matrix_power(A_c, k) @ B for k in range(A_c.shape[0])])
print("controllability matrix:\n", ctrb)
print("rank:", np.linalg.matrix_rank(ctrb), "of", A_c.shape[0], "-> controllable:",
      np.linalg.matrix_rank(ctrb) == A_c.shape[0])

---
## 12. Random numbers, noise, and reproducibility

Use `np.random.default_rng(seed)`. The legacy `np.random.seed` / `np.random.randn` interface
still works but shares one global state, which makes results hard to reproduce once you have
more than one source of randomness.

In [ ]:
rng = np.random.default_rng(42)

n = 10_000
white = rng.normal(0, 1, n)                # Gaussian white noise
uniform = rng.uniform(-1, 1, n)
impulsive = rng.laplace(0, 0.5, n)         # heavier tails -- models clicks and pops

fig, ax = plt.subplots(1, 3, figsize=(12, 2.6))
for a, (data, name) in zip(ax, [(white, "Gaussian"), (uniform, "Uniform"), (impulsive, "Laplace")]):
    a.hist(data, bins=60, density=True, alpha=0.8)
    a.set_title(f"{name}  (σ = {data.std():.2f})")
fig.tight_layout()

# Reproducibility: same seed, same numbers, always.
print(np.random.default_rng(42).normal(size=3))
print(np.random.default_rng(42).normal(size=3))

In [ ]:
# "White" means flat on average across frequency. One realisation is very noisy;
# averaging many realisations reveals the flat spectrum.
fs, N = 1000.0, 1024
f = np.fft.rfftfreq(N, 1 / fs)

single = np.abs(np.fft.rfft(rng.normal(size=N))) ** 2
averaged = np.mean([np.abs(np.fft.rfft(rng.normal(size=N))) ** 2 for _ in range(500)], axis=0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(f, single, lw=0.5, alpha=0.6, label="one realisation")
ax.semilogy(f, averaged, lw=1.5, label="average of 500")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("power")
ax.legend(); ax.set_title("White noise is flat only in expectation")

---
## 13. Saving and loading

`.npy` for a single array, `.npz` for several, text formats when a human or another tool
needs to read it.

In [ ]:
import os
os.makedirs("../scratch", exist_ok=True)

t, N = time_vector(500, 1.0)
x = np.sin(2 * np.pi * 5 * t)

np.save("../scratch/signal.npy", x)                          # binary, exact, fast
np.savez("../scratch/capture.npz", t=t, x=x, fs=500.0)       # several arrays
np.savetxt("../scratch/signal.csv", np.column_stack([t, x]),
           delimiter=",", header="time_s,amplitude", comments="")

back = np.load("../scratch/signal.npy")
bundle = np.load("../scratch/capture.npz")

print("round-trip exact:", np.array_equal(x, back))
print("npz keys        :", bundle.files)
print("fs from npz     :", bundle["fs"])

In [ ]:
# Reading the repository's own data file.
data = np.loadtxt("../data/ecg_like.csv", delimiter=",", skiprows=1)
t_ecg, mv = data[:, 0], data[:, 1]

print("shape:", data.shape, " duration:", t_ecg[-1].round(2), "s")
print("estimated fs:", round(1 / np.mean(np.diff(t_ecg))), "Hz")

plt.figure(figsize=(10, 2.8))
plt.plot(t_ecg[:1800], mv[:1800], lw=0.8)
plt.xlabel("time [s]"); plt.ylabel("mV")
plt.title("ecg_like.csv -- note the slow baseline wander and the 50 Hz hum")

---
## 14. Exercises

Work these in new cells below. No solutions are provided — checking your own answer
numerically *is* the skill.

**1. Build a signal generator.** Write `make_signal(fs, duration, components)` where
`components` is a list of `(amplitude, frequency, phase)` tuples. Return `(t, x)`. Verify
with an FFT that each requested component appears at the right frequency and amplitude.

**2. Prove Parseval's theorem.** For a random real signal, show numerically that
$\sum_n |x[n]|^2 = \frac{1}{N}\sum_k |X[k]|^2$. Get the factor of $N$ right — that is the
whole exercise.

**3. Frequency resolution.** Take a signal containing 50 Hz and 52 Hz. Sample at 1 kHz and
capture for 0.2 s, 0.5 s, and 2 s. At which duration can you resolve the two peaks? Relate
your answer to $\Delta f = 1/T$.

**4. Write your own convolution.** Implement `my_convolve(x, h)` with explicit loops, matching
`np.convolve(x, h, mode='full')`. Then time both. The ratio tells you what NumPy buys you.

**5. Echo and de-echo.** Add an echo: $y[n] = x[n] + 0.6\,x[n-2000]$ at $f_s = 8$ kHz. Use
autocorrelation to recover the delay from $y$ alone, then design an inverse filter
$y[n] - 0.6\,\hat{y}[n-2000]$ to remove it. Compare SNR before and after.

**6. Aliasing hunt.** With $f_s = 200$ Hz, find three distinct input frequencies above
Nyquist that all alias to exactly 30 Hz. Verify by sampling each and comparing the sample
values.

**7. Load `bench_capture.txt` with NumPy alone.** Skip the comment lines, handle the `;`
delimiter, and deal with the `NaN` and `-999.0` sentinel values. (Notebook 04 does this
comfortably in pandas — doing it in raw NumPy first shows you what pandas is saving you from.)

**8. Second-order system from scratch.** For $\zeta = 0.2$ and $\omega_n = 10$, compute the
step response two ways: the analytic formula, and by numerically convolving the impulse
response with a unit step. Plot both and quantify the difference.

---

### Where to go next

- **`02_matplotlib_for_signals.ipynb`** — presenting all of this properly: stem plots, Bode
  plots, pole-zero maps, spectrograms.
- **`03_scipy_signal_toolbox.ipynb`** — the tools that replace most of the hand-rolled code
  above: real filter design, LTI objects, Welch spectra, resampling.

### Quick reference

| Task | Call |
|------|------|
| time vector | `np.arange(N) / fs` |
| complex exponential | `np.exp(1j * 2 * np.pi * f * t)` |
| unit step | `(t >= t0).astype(float)` |
| convolution | `np.convolve(x, h, mode=...)` |
| correlation | `np.correlate(x, y, mode='full')` |
| real FFT | `np.fft.rfft(x)`, `np.fft.rfftfreq(N, 1/fs)` |
| inverse | `np.fft.irfft(X, n=N)` |
| magnitude / phase | `np.abs(X)`, `np.angle(X)` |
| window | `np.hanning(N)`, `np.hamming(N)`, `np.blackman(N)` |
| RMS | `np.sqrt(np.mean(x**2))` |
| eigenvalues | `np.linalg.eigvals(A)` |
| reproducible noise | `np.random.default_rng(seed).normal(...)` |